# # Phases 7–8 — Uncertainty, Practical Design Tool, Ablations
# Rock Fragmentation Prediction from Blast Parameters
#
# This notebook needs the **outputs of Notebook 02** (`oof_P2_rgkf.csv`,
# `oof_P3_loso.csv`) and, for the design-tool demo, the fitted models from
# **Notebook 03** (`models/final_models.joblib`). Add both as Kaggle Dataset
# inputs alongside `blast_master_110.csv` (e.g.
# `rock-blast-benchmark-outputs`).

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
IN_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("../data/processed")
PREV_OUT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("../outputs")  # Notebook 02/03 outputs
CSV = next(IN_ROOT.rglob("blast_master_110.csv"))
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
for sub in ["figures", "tables", "models"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV)
FEATURES = ["S_B", "H_B", "B_D", "T_B", "Pf", "XB", "E"]
TARGET = "X50"

oof_P2 = pd.read_csv(next(PREV_OUT.rglob("oof_P2_rgkf.csv")))
oof_P3 = pd.read_csv(next(PREV_OUT.rglob("oof_P3_loso.csv")))
final = joblib.load(next(PREV_OUT.rglob("final_models.joblib")))
print("Loaded OOF predictions and", list(final.keys()), "fitted models")

# ## 4.1 Split-conformal prediction intervals (Phase 7)
# We estimate a 90% prediction interval half-width as the (1−α)-quantile of
# |residual| on a calibration half of the out-of-fold predictions, evaluated
# on the other half, averaged over many random splits. Empirical coverage
# should land close to 90% under P2 (in-distribution folds); a drop under
# P3 (LOSO) would indicate the model is less reliable on unseen sites — an
# expected and important finding to report.

In [ ]:
def conformal_coverage(oof, model, alpha=0.10, n_iter=200, seed=SEED):
    d = oof[(oof.model == model) & (oof.rep == oof.rep.iloc[0])]
    r = np.abs(d.y - d.pred).values
    rng = np.random.RandomState(seed)
    cover, width = [], []
    for _ in range(n_iter):
        perm = rng.permutation(len(r))
        cal, ev = perm[: len(r) // 2], perm[len(r) // 2:]
        n = len(cal)
        q = np.quantile(r[cal], min(1, np.ceil((n + 1) * (1 - alpha)) / n))
        cover.append(np.mean(r[ev] <= q))
        width.append(2 * q)
    return round(float(np.mean(cover)), 3), round(float(np.mean(width)), 3)


rows = []
for m in oof_P2.model.unique():
    c2, w2 = conformal_coverage(oof_P2, m)
    c3, w3 = conformal_coverage(oof_P3, m) if m in oof_P3.model.unique() else (np.nan, np.nan)
    rows.append({"model": m, "P2_coverage": c2, "P2_interval_width_m": w2,
                 "P3_coverage": c3, "P3_interval_width_m": w3})
conf = pd.DataFrame(rows).sort_values("P2_interval_width_m")
conf.to_csv(OUT / "tables/T8_conformal_coverage.csv", index=False)
conf

# ## 4.2 Applicability domain
# A new blast design is flagged as **extrapolation** if any input falls
# outside the training min–max range, or if its Mahalanobis distance to the
# training-data centroid exceeds the 97.5th-percentile distance observed in
# the training set itself.

In [ ]:
from scipy.spatial.distance import mahalanobis

Xtr = df[FEATURES].values
mu, cov_inv = Xtr.mean(0), np.linalg.pinv(np.cov(Xtr.T))
train_d = np.array([mahalanobis(x, mu, cov_inv) for x in Xtr])
MAHAL_THRESH = np.percentile(train_d, 97.5)
RANGE = df[FEATURES].agg(["min", "max"])


def in_domain(x_dict):
    x = np.array([x_dict[f] for f in FEATURES])
    range_ok = all(RANGE.loc["min", f] <= x_dict[f] <= RANGE.loc["max", f] for f in FEATURES)
    dist_ok = mahalanobis(x, mu, cov_inv) <= MAHAL_THRESH
    return range_ok and dist_ok


print("Training-range box:\n", RANGE)
print(f"Mahalanobis 97.5th-percentile threshold: {MAHAL_THRESH:.2f}")

# ## 4.3 Practical design-tool demo (Figure 9)
# For a representative site (Akdaglar quarry, median geometry) we sweep the
# powder factor `Pf` and plot the predicted X50 with its 90% conformal
# interval — the kind of chart a blasting engineer could use to select `Pf`
# for a target fragment size.

In [ ]:
import matplotlib.pyplot as plt

best_model_name = "XGB"
q90 = np.quantile(np.abs(oof_P2.query("model == @best_model_name").y
                          - oof_P2.query("model == @best_model_name").pred), 0.9)

base = df[df.site == "Akdaglar"][FEATURES].median()
grid = pd.DataFrame([base] * 50)
grid["Pf"] = np.linspace(df.Pf.min(), df.Pf.max(), 50)
grid["in_domain"] = grid.apply(lambda r: in_domain(r[FEATURES].to_dict()), axis=1)
pred = final[best_model_name].predict(grid[FEATURES])

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(grid.Pf, pred, color="#3b6ea5", label=f"{best_model_name} prediction")
ax.fill_between(grid.Pf, pred - q90, pred + q90, alpha=0.25, color="#3b6ea5",
                 label="90% conformal interval")
ax.scatter(df[df.site == "Akdaglar"].Pf, df[df.site == "Akdaglar"].X50,
           color="black", s=25, zorder=5, label="Observed Akdaglar blasts")
ax.set_xlabel("Powder factor (kg/m$^3$)")
ax.set_ylabel("Predicted X50 (m)")
ax.set_title("Akdaglar quarry, median blast geometry")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures/F9_design_tool.png", dpi=300)
plt.show()

# ## 4.4 Ablation studies (Phase 8)
# Reuses the evaluation machinery from Notebook 02. Paste `metrics`,
# `repeated_group_kfold`, `build`, `space`, `tune`, `run_protocol` from
# Notebook 02 §2.2–2.5 above this cell if running standalone.

In [ ]:
# --- minimal re-import of what's needed for ablations (see Notebook 02) ---
import time

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {"R2": r2_score(y_true, y_pred), "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAE": mean_absolute_error(y_true, y_pred),
            "MAPE": np.mean(np.abs((y_true - y_pred) / y_true)) * 100}


def logy(est):
    return TransformedTargetRegressor(regressor=est, func=np.log, inverse_func=np.exp)


def build_abl(name, p):
    if name == "MVR":
        return logy(make_pipeline(FunctionTransformer(np.log), LinearRegression()))
    if name == "RF":
        return RandomForestRegressor(random_state=SEED, n_jobs=-1, **p)
    if name == "XGB":
        return XGBRegressor(random_state=SEED, n_jobs=4, verbosity=0, **p)
    if name == "SVR":
        return logy(make_pipeline(StandardScaler(), SVR(**p)))


def space_abl(name, t):
    if name == "RF":
        return {"n_estimators": t.suggest_int("n_estimators", 200, 800, step=100),
                "max_depth": t.suggest_int("max_depth", 2, 12),
                "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 8)}
    if name == "XGB":
        return {"n_estimators": t.suggest_int("n_estimators", 100, 600, step=50),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "max_depth": t.suggest_int("max_depth", 2, 6)}
    if name == "SVR":
        return {"C": t.suggest_float("C", 0.1, 1000, log=True),
                "epsilon": t.suggest_float("epsilon", 1e-3, 0.3, log=True),
                "gamma": t.suggest_float("gamma", 1e-3, 10, log=True)}
    return {}


def tune_abl(name, feats, Xtr, ytr, gtr, n_trials=20):
    if name == "MVR":
        return {}
    cv = GroupKFold(n_splits=5)

    def obj(t):
        est = build_abl(name, space_abl(name, t))
        return -cross_val_score(est, Xtr, ytr, groups=gtr, cv=cv,
                                 scoring="neg_root_mean_squared_error").mean()

    st = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    st.optimize(obj, n_trials=n_trials, show_progress_bar=False)
    return space_abl(name, optuna.trial.FixedTrial(st.best_params))


def run_ablation(feats, label, n_repeats=5):
    Xa = df[feats].values
    ya = df[TARGET].values
    ga = df.groupby(feats, sort=False).ngroup().values
    rows = []
    rng = np.random.RandomState(SEED)
    uniq = np.unique(ga)
    for rep in range(n_repeats):
        perm = rng.permutation(uniq)
        for fold, fg in enumerate(np.array_split(perm, 5)):
            te = np.isin(ga, fg)
            tr = ~te
            for name in ["MVR", "RF", "XGB", "SVR"]:
                p = tune_abl(name, feats, Xa[tr], ya[tr], ga[tr])
                est = build_abl(name, p).fit(Xa[tr], ya[tr])
                pred = est.predict(Xa[te])
                rows.append({"ablation": label, "rep": rep, "fold": fold,
                             "model": name, **metrics(ya[te], pred)})
    return pd.DataFrame(rows)


FEATURE_SETS = {
    "geometry_only": ["S_B", "H_B", "B_D", "T_B"],
    "geometry+Pf": ["S_B", "H_B", "B_D", "T_B", "Pf"],
    "all_7_features": FEATURES,
    "no_E_(site_proxy)": [f for f in FEATURES if f != "E"],
    "no_B_D": [f for f in FEATURES if f != "B_D"],
}
abl_rows = []
for label, feats in FEATURE_SETS.items():
    print("Ablation:", label, feats)
    abl_rows.append(run_ablation(feats, label))
ablation_results = pd.concat(abl_rows, ignore_index=True)
ablation_results.to_csv(OUT / "tables/ablation_raw.csv", index=False)

ablation_summary = (ablation_results.groupby(["ablation", "model"])[["R2", "RMSE", "MAE", "MAPE"]]
                     .agg(["mean", "std"]).round(4))
ablation_summary.to_csv(OUT / "tables/T6_ablations.csv")
ablation_summary

# ## 4.5 Additional robustness checks (Phase 8, quick versions)
# - **With vs. without Miami Mine (7 blasts)** — compares against studies
#   built on the 102/103-blast subset of the database.
# - **With vs. without Isolation-Forest-flagged rows** (from Notebook 01 §1.8).

In [ ]:
from sklearn.ensemble import IsolationForest

iso_mask = IsolationForest(random_state=SEED, contamination=0.05).fit_predict(
    df[FEATURES + [TARGET]]) == -1

subsets = {
    "full_110": df.index.values,
    "without_Miami_103": df[df.site != "Miami"].index.values,
    "without_iso_outliers": df[~iso_mask].index.values,
}
robust_rows = []
for label, idx in subsets.items():
    sub = df.loc[idx]
    feats_df = sub[FEATURES]
    ya = sub[TARGET].values
    ga = sub.groupby(FEATURES, sort=False).ngroup().values
    rng = np.random.RandomState(SEED)
    uniq = np.unique(ga)
    for fold, fg in enumerate(np.array_split(rng.permutation(uniq), 5)):
        te = np.isin(ga, fg)
        tr = ~te
        for name in ["RF", "XGB"]:
            p = tune_abl(name, FEATURES, feats_df.values[tr], ya[tr], ga[tr], n_trials=15)
            est = build_abl(name, p).fit(feats_df.values[tr], ya[tr])
            pred = est.predict(feats_df.values[te])
            robust_rows.append({"subset": label, "fold": fold, "model": name, **metrics(ya[te], pred)})
robust = pd.DataFrame(robust_rows)
robust.to_csv(OUT / "tables/robustness_subsets.csv", index=False)
robust.groupby(["subset", "model"])[["R2", "RMSE"]].mean().round(4)

# ## Outputs of this notebook
# - `tables/T8_conformal_coverage.csv`, `T6_ablations.csv`,
#   `robustness_subsets.csv`
# - `figures/F9_design_tool.png`
#
# All four notebooks together produce every table and figure referenced in
# `paper/paper.tex` (see `paper/make_tables.py` to auto-generate the LaTeX
# table files from these CSVs).